In [4]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import fisher_exact, chi2_contingency, mannwhitneyu
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.multitest import multipletests

# 파일 경로 설정
promo_0_path = Path(r"C:\myCode\ott-churn-prediction\kim.kwangil\derived_variable\260510_user_features_0.csv")
promo_1_path = Path(r"C:\myCode\ott-churn-prediction\kim.kwangil\derived_variable\260510_user_features_1.csv")

# 기준 설정
alpha = 0.05
confidence_level = int((1 - alpha) * 100)

# 다중비교 보정 사용 여부 설정
# True  : 나이대별 컬럼 검정 결과에 대해 FDR 보정 적용
# False : 단순 p < 0.05 기준 적용
use_fdr = True

age_labels = ["10대", "20대", "30대", "40대", "50대", "60대", "70대"]

# 최종 모델 60개 기준 설정
# 62개 전체를 보려면 exclude_features = [] 로 변경
exclude_features = ["gender", "payment_device"]

# 사용 컬럼 정의
basic_features = [
    "age",
    "payment_device",
    "gender",
]

mem_features = [
    "mem_tenure_days",
    "mem_billing_method_value",
    "mem_screen_2_flag",
    "mem_is_verified",
    "mem_is_female",
    "mem_is_male",
    "mem_reg_hour_afternoon",
    "mem_reg_hour_evening",
    "mem_reg_hour_night",
    "mem_reg_weekday",
    "mem_reg_is_weekend",
    "mem_verified_multi_screen",
    "mem_verified_premium_screen",
    "mem_billing_method_131_flag",
    "mem_billing_method_132_flag",
    "mem_billing_method_140_flag",
    "mem_billing_method_151_flag",
    "mem_billing_method_170_flag",
    "mem_billing_method_180_flag",
    "mem_device_mobile_flag",
    "mem_device_pc_flag",
    "mem_device_smarttv_flag",
]

usage_intensity_features = [
    "vh_median_watch_min",
    "vh_std_watch_min",
    "vh_max_watch_min",
    "vh_short_watch_ratio",
    "vh_max_daily_events",
    "vh_multi_event_day_ratio",
    "vh_std_daily_watch_min",
    "vh_titles_per_active_day",
    "vh_watch_min_per_active_day",
    "vh_watch_min_per_tenure_day",
    "vh_rewatch_event_ratio",
    "vh_repeat_event_count",
    "vh_avg_watch_min_per_title",
    "vh_activity_density",
    "vh_binge_index",
]

time_pattern_features = [
    "vh_weekend_ratio",
    "vh_week2_watch_ratio",
    "vh_week3_watch_ratio",
    "vh_w2_minus_w1_watch_min",
    "vh_w3_minus_w2_watch_min",
    "vh_w3_to_w1_ratio_capped",
]

content_features = [
    "vh_recent_release_180d_ratio",
    "vh_recent_release_365d_ratio",
    "vh_old_catalog_5y_ratio",
    "vh_median_content_age_days",
    "vh_genre_unique_count",
    "vh_top_genre_share",
]

genre_share_features = [
    "genre_share__Action_Adventure",
    "genre_share__Animation_Family",
    "genre_share__Comedy",
    "genre_share__Drama",
    "genre_share__Historical_War",
    "genre_share__Horror",
    "genre_share__Other",
    "genre_share__Romance",
    "genre_share__SF_Fantasy",
    "genre_share__Thriller_Crime",
]

used_features = (
    basic_features
    + mem_features
    + usage_intensity_features
    + time_pattern_features
    + content_features
    + genre_share_features
)

used_features = [col for col in used_features if col not in exclude_features]

# 컬럼 타입 정의
binary_features = {
    "mem_screen_2_flag",
    "mem_is_verified",
    "mem_is_female",
    "mem_is_male",
    "mem_reg_hour_afternoon",
    "mem_reg_hour_evening",
    "mem_reg_hour_night",
    "mem_reg_is_weekend",
    "mem_verified_multi_screen",
    "mem_verified_premium_screen",
    "mem_billing_method_131_flag",
    "mem_billing_method_132_flag",
    "mem_billing_method_140_flag",
    "mem_billing_method_151_flag",
    "mem_billing_method_170_flag",
    "mem_billing_method_180_flag",
    "mem_device_mobile_flag",
    "mem_device_pc_flag",
    "mem_device_smarttv_flag",
}

categorical_features = {
    "payment_device",
    "gender",
    "mem_billing_method_value",
    "mem_reg_weekday",
}

def check_input_files():
    missing_paths = [str(path) for path in [promo_0_path, promo_1_path] if not path.exists()]
    if missing_paths:
        raise FileNotFoundError(
            "아래 파일이 존재하지 않습니다.\n" + "\n".join(missing_paths)
        )

def load_data():
    check_input_files()
    df_promo_0 = pd.read_csv(promo_0_path)
    df_promo_1 = pd.read_csv(promo_1_path)
    return df_promo_0, df_promo_1

def to_binary(series):
    lowered = series.astype(str).str.strip().str.lower()

    mapped = lowered.map(
        {
            "1": 1,
            "0": 0,
            "y": 1,
            "n": 0,
            "yes": 1,
            "no": 0,
            "true": 1,
            "false": 0,
        }
    )

    numeric = pd.to_numeric(series, errors="coerce")
    return mapped.where(mapped.notna(), numeric)

def preprocess_for_churn(df):
    temp = df.copy()

    # 필수 컬럼 정리
    temp = temp[temp["age"].notna() & temp["is_repurchase"].notna()].copy()

    # age 숫자형 변환
    temp["age"] = pd.to_numeric(temp["age"], errors="coerce")
    temp = temp[temp["age"].notna()].copy()

    # 나이대 생성
    temp["age_group"] = pd.cut(
        temp["age"],
        bins=[10, 20, 30, 40, 50, 60, 70, 80],
        labels=age_labels,
        right=False
    )
    temp = temp[temp["age_group"].notna()].copy()

    # 이탈 컬럼 생성
    temp["is_repurchase_num"] = to_binary(temp["is_repurchase"])
    temp = temp[temp["is_repurchase_num"].isin([0, 1])].copy()
    temp["churn"] = (temp["is_repurchase_num"].astype(int) == 0).astype(int)

    return temp

def run_proportion_test(success_0, total_0, success_1, total_1):
    nonsuccess_0 = total_0 - success_0
    nonsuccess_1 = total_1 - success_1

    small_sample_flag = (
        min(success_0, nonsuccess_0, success_1, nonsuccess_1) < 5
        or min(total_0, total_1) < 30
    )

    if small_sample_flag:
        _, pvalue = fisher_exact(
            [[success_0, nonsuccess_0], [success_1, nonsuccess_1]],
            alternative="two-sided"
        )
    else:
        _, pvalue = proportions_ztest(
            count=[success_0, success_1],
            nobs=[total_0, total_1]
        )

    return float(pvalue)

def find_significant_age_groups(df0, df1):
    churn_df0 = preprocess_for_churn(df0)
    churn_df1 = preprocess_for_churn(df1)

    g0 = (
        churn_df0.groupby("age_group", observed=True)["churn"]
        .agg(churn_0="sum", total_0="count")
        .reset_index()
    )

    g1 = (
        churn_df1.groupby("age_group", observed=True)["churn"]
        .agg(churn_1="sum", total_1="count")
        .reset_index()
    )

    merged = pd.merge(g0, g1, on="age_group", how="inner")
    merged = merged[(merged["total_0"] > 0) & (merged["total_1"] > 0)].copy()

    significant_age_groups = []

    for _, row in merged.iterrows():
        pvalue = run_proportion_test(
            success_0=int(row["churn_0"]),
            total_0=int(row["total_0"]),
            success_1=int(row["churn_1"]),
            total_1=int(row["total_1"])
        )

        if pvalue < alpha:
            significant_age_groups.append(row["age_group"])

    significant_age_groups = [age for age in age_labels if age in significant_age_groups]
    return significant_age_groups

def preprocess_for_feature_test(df, target_age_groups):
    temp = df.copy()

    # 필수 컬럼 정리
    temp = temp[temp["age"].notna() & temp["is_repurchase"].notna()].copy()

    # age 숫자형 변환
    temp["age"] = pd.to_numeric(temp["age"], errors="coerce")
    temp = temp[temp["age"].notna()].copy()

    # 나이대 생성
    temp["age_group"] = pd.cut(
        temp["age"],
        bins=[10, 20, 30, 40, 50, 60, 70, 80],
        labels=age_labels,
        right=False
    )
    temp = temp[temp["age_group"].notna()].copy()

    # 대상 나이대 필터링
    temp = temp[temp["age_group"].isin(target_age_groups)].copy()

    return temp

def normalize_binary_series(series):
    temp = series.copy()

    temp = temp.replace(
        {
            True: 1,
            False: 0,
            "Y": 1,
            "N": 0,
            "y": 1,
            "n": 0,
        }
    )

    temp = pd.to_numeric(temp, errors="coerce")
    temp = temp[temp.isin([0, 1])].astype(int)

    return temp

def detect_feature_type(feature_name):
    if feature_name in binary_features:
        return "binary"
    if feature_name in categorical_features:
        return "categorical"
    return "numeric"

def run_binary_test(series_0, series_1):
    s0 = normalize_binary_series(series_0)
    s1 = normalize_binary_series(series_1)

    total_0 = len(s0)
    total_1 = len(s1)

    if total_0 == 0 or total_1 == 0:
        return np.nan

    success_0 = int(s0.sum())
    success_1 = int(s1.sum())
    nonsuccess_0 = total_0 - success_0
    nonsuccess_1 = total_1 - success_1

    if (
        success_0 == success_1
        and nonsuccess_0 == nonsuccess_1
        and pd.concat([s0, s1]).nunique() == 1
    ):
        return 1.0

    return run_proportion_test(success_0, total_0, success_1, total_1)

def run_categorical_test(series_0, series_1):
    s0 = series_0.dropna().astype(str)
    s1 = series_1.dropna().astype(str)

    if len(s0) == 0 or len(s1) == 0:
        return np.nan

    group = pd.Series(["promo_0"] * len(s0) + ["promo_1"] * len(s1), name="group")
    value = pd.concat([s0, s1], ignore_index=True)
    contingency = pd.crosstab(group, value, dropna=False)

    if contingency.shape[1] <= 1:
        return 1.0

    _, pvalue, _, expected = chi2_contingency(contingency)
    low_expected_cell_count = int((expected < 5).sum())

    if contingency.shape == (2, 2) and low_expected_cell_count > 0:
        _, pvalue = fisher_exact(contingency.values, alternative="two-sided")

    return float(pvalue)

def run_numeric_test(series_0, series_1):
    s0 = pd.to_numeric(series_0, errors="coerce").dropna()
    s1 = pd.to_numeric(series_1, errors="coerce").dropna()

    if len(s0) == 0 or len(s1) == 0:
        return np.nan

    if pd.concat([s0, s1]).nunique() <= 1:
        return 1.0

    _, pvalue = mannwhitneyu(
        s0,
        s1,
        alternative="two-sided",
        method="auto"
    )
    return float(pvalue)

def get_feature_pvalue(df0_age, df1_age, feature_name):
    if feature_name not in df0_age.columns or feature_name not in df1_age.columns:
        return np.nan

    feature_type = detect_feature_type(feature_name)
    series_0 = df0_age[feature_name]
    series_1 = df1_age[feature_name]

    if feature_type == "binary":
        return run_binary_test(series_0, series_1)
    if feature_type == "categorical":
        return run_categorical_test(series_0, series_1)
    return run_numeric_test(series_0, series_1)

def count_significant_features_by_age_group(df0, df1, target_age_groups):
    feature_df0 = preprocess_for_feature_test(df0, target_age_groups)
    feature_df1 = preprocess_for_feature_test(df1, target_age_groups)

    actual_features = [
        col for col in used_features
        if col in feature_df0.columns and col in feature_df1.columns
    ]

    result_rows = []

    for age_group in target_age_groups:
        df0_age = feature_df0[feature_df0["age_group"] == age_group].copy()
        df1_age = feature_df1[feature_df1["age_group"] == age_group].copy()

        pvalues = []

        for feature_name in actual_features:
            pvalue = get_feature_pvalue(df0_age, df1_age, feature_name)

            if pd.notna(pvalue):
                pvalues.append(pvalue)

        if not pvalues:
            significant_count = 0
        elif use_fdr:
            reject, _, _, _ = multipletests(
                pvalues,
                alpha=alpha,
                method="fdr_bh"
            )
            significant_count = int(reject.sum())
        else:
            significant_count = int(np.sum(np.array(pvalues) < alpha))

        result_rows.append(
            {
                "age_group": age_group,
                "significant_feature_count": significant_count
            }
        )

    result_df = pd.DataFrame(result_rows)
    return result_df, actual_features

# 실행 구간
df_promo_0, df_promo_1 = load_data()

significant_age_groups = find_significant_age_groups(df_promo_0, df_promo_1)

print(f"{confidence_level}% 신뢰수준 기준 자동 탐지 유의 나이대")

if not significant_age_groups:
    print("유의한 나이대 없음")
else:
    print(", ".join(significant_age_groups))
    print()

    count_df, actual_features = count_significant_features_by_age_group(
        df_promo_0,
        df_promo_1,
        significant_age_groups
    )

    total_feature_count = len(actual_features)

    print("각 유의 나이대에서 프로모션 여부에 따라 유의하게 차이 나는 컬럼 개수")

    for _, row in count_df.iterrows():
        print(
            f"{row['age_group']}: "
            f"총 {total_feature_count}개 컬럼 중 "
            f"{row['significant_feature_count']}개"
        )


95% 신뢰수준 기준 자동 탐지 유의 나이대
10대, 20대, 30대, 40대, 60대

각 유의 나이대에서 프로모션 여부에 따라 유의하게 차이 나는 컬럼 개수
10대: 총 60개 컬럼 중 11개
20대: 총 60개 컬럼 중 23개
30대: 총 60개 컬럼 중 16개
40대: 총 60개 컬럼 중 19개
60대: 총 60개 컬럼 중 6개
